In [1]:
import os
os.chdir('/workspace/0e92e95e-936a-46e4-8e27-f8cba263044b')
print(os.listdir('.'))


['memory', '.prompts', '.config', '-PROMPT-v6-DATASET.md', 'lmfdb_Lchi5.json', '.kernel_llm_logs_1.txt']


In [2]:
import json
with open('lmfdb_Lchi5.json','r') as f:
 lmfdb = json.load(f)
print(type(lmfdb))
if isinstance(lmfdb, dict):
 print(list(lmfdb.keys())[:20])
elif isinstance(lmfdb, list):
 print(len(lmfdb), lmfdb[:5])


JSONDecodeError: Expecting value: line 2 column 1 (char 1)

In [3]:
with open('lmfdb_Lchi5.json','r') as f:
 raw = f.read()
print(len(raw))
print(raw[:500])


9803

# Zeros of 1-5-5.4-r0-0-0 downloaded from the LMFDB on 30 May 2026.
{"order_of_vanishing": 0, "positive_zeros": ["6.64845334472771471612327845997", "9.831444432886669616348321347458", "11.95884562608351453026565868826", "16.03382112838423567459325378224", "17.566994292325555202701595268144", "19.54073262278475025037869002299", "22.227405454459410911877624963081", "24.58846621740819520765626997608", "26.77609594800414011652357496527", "28.46103510017752247518697827232", "29.707909350480965569230


In [4]:
lines = raw.splitlines()
# Find first JSON line
json_text = '\n'.join(l for l in lines if not l.startswith('#'))
data = json.loads(json_text)
print(list(data.keys()))
print('num zeros:', len(data['positive_zeros']))
print('first 5:', data['positive_zeros'][:5])
print('last 3:', data['positive_zeros'][-3:])


['order_of_vanishing', 'positive_zeros', 'negative_zeros', 'positive_zeros_accuracy', 'negative_zeros_accuracy']
num zeros: 129
first 5: ['6.64845334472771471612327845997', '9.831444432886669616348321347458', '11.95884562608351453026565868826', '16.03382112838423567459325378224', '17.566994292325555202701595268144']
last 3: ['196.54076456244967267018733746529', '197.86100950772366771521398079814', '199.09102170624125905531999800239']


In [5]:
print(data['positive_zeros_accuracy'], data['order_of_vanishing'])


None 0


In [6]:

# Let's understand the setup. Character: real primitive quadratic mod 5.
# chi5(n) = Legendre symbol (n/5) extended periodically: chi(0)=0, chi(1)=1, chi(2)=-1, chi(3)=-1, chi(4)=1.
# This is a real, even (chi(-1)=chi(4)=1) primitive Dirichlet character mod 5.
# Wait: chi(-1) = chi(4 mod 5) = 1, so chi is even (a=0 in gamma factor).
# Hardy Z-function for even primitive character chi mod q (a=0):
# Z(t) = exp(i*theta(t)) * L(1/2 + i*t, chi), where
# theta(t) = arg(Gamma((1/2 + i t)/2)) - (t/2) * log(pi/q) + arg of root number factor... 
# Actually completed function: Lambda(s, chi) = (q/pi)^(s/2) * Gamma((s+a)/2) * L(s, chi) 
# satisfies Lambda(s, chi) = epsilon * Lambda(1-s, chi-bar). For real chi, chi-bar = chi.
# root number epsilon = +1 for chi_5 (it's the Kronecker symbol mod 5, q=5).
# So Z(t) = exp(i*theta(t)) * L(1/2 + i*t, chi) is real.
# theta(t) = arg Gamma((1/2 + i*t + a)/2) - (t/2) log(pi/q)
# For a=0:
# theta(t) = arg Gamma(1/4 + i t/2) - (t/2) log(pi/5)
# Let me verify with the 5.4 character. LMFDB label 5.4 means modulus 5, Conrey index 4.
# Conrey 5.4 = chi_4 mod 5, which is the unique real character: maps generator 2 -> -1.
# Indeed legendre symbol (2/5) = -1. So this is the legendre symbol chi. Real, even (chi(-1)=1).

# Now first zero in LMFDB: 6.64845... 
# Let's verify by computing L(1/2 + i*6.64845, chi).

import mpmath as mp
mp.mp.dps = 50

def chi5(n):
 n = n % 5
 return {0:0, 1:1, 2:-1, 3:-1, 4:1}[n]

def Lchi5(s, terms=200):
 # Use functional equation / mpmath's dirichlet
 # Better: mpmath has mp.dirichlet
 return mp.dirichlet(s, [chi5(0), chi5(1), chi5(2), chi5(3), chi5(4)])

# Test at first zero
s = mp.mpc('0.5', '6.64845334472771471612327845997')
print('L at first zero:', Lchi5(s))


L at first zero: (-3.818042697809341098693732377105952040328876704847e-30 - 1.4743687960003469730198830667496115207334935534232e-29j)


In [7]:

# Excellent — mp.dirichlet works perfectly. Now let's define a Hardy Z function.
# For even real primitive chi mod q, root number = +1:
# Z(t) = exp(i*theta(t)) * L(1/2 + it, chi)
# theta(t) = arg(Gamma(1/4 + it/2)) - (t/2)*log(pi/q)

def theta_chi5(t):
 return mp.arg(mp.gamma(mp.mpf('0.25') + 1j*t/2)) - (t/2)*mp.log(mp.pi/5)

def Z_chi5(t):
 s = mp.mpc('0.5', t)
 L = mp.dirichlet(s, [0,1,-1,-1,1])
 return (mp.exp(1j*theta_chi5(t)) * L).real

# Test: Z near zero should change sign at 6.648...
mp.mp.dps = 30
for t in [6.0, 6.5, 6.65, 6.7, 7.0]:
 print(t, Z_chi5(mp.mpf(t)))


6.0 1.03282597118493695109253093991
6.5 0.24326001259148911848196765677
6.65 -0.0025278719239209466070896587613
6.7 -0.084073814054956704946586131077
7.0 -0.55869358601774049098932542445


In [8]:

# Z(t) sign changes at 6.648. Good.
# Now estimate: for the 5000th zero, we need to estimate where it lies.
# Riemann-von Mangoldt-like formula: N(T) = (T/pi)*log(qT/(2*pi*e)) + O(log T)
# For chi mod q=5 (a=0): N(T) ~ (T/pi)*log(qT/(2*pi e))
# We want N(T)=5000. Solve approximately:
from scipy.optimize import brentq
import math
def Napprox(T, q=5):
 return (T/math.pi)*math.log(q*T/(2*math.pi*math.e))
for T in [1000, 1500, 2000, 2500, 3000]:
 print(T, Napprox(T))


1000 1807.782469352711
1500 2905.269032649179
2000 4056.836139010725
2500 5248.617169817089
3000 6472.4448657563125


In [9]:

# So we need T up to about 2400-2450 for N=5000. Let's use a safety margin: scan up to T=2500.
T_target = brentq(lambda T: Napprox(T)-5000, 100, 5000)
print('T for N=5000:', T_target)
T_scan = 2500 # safety margin
print('Scan up to T=', T_scan)


T for N=5000: 2396.8864206352396
Scan up to T= 2500


In [10]:

# Strategy:
# 1) Use a low working precision (say dps=15-25) for the sign-change scan (fast).
# 2) Parallelize via multiprocessing.
# 3) For each sign change interval, refine to dps=50 using mpmath.findroot.
# 4) Validate first 129 zeros against LMFDB.
#
# Step size: average gap between zeros at height T ~ 2pi/log(qT/(2pi)).
# At T=100: gap ~ 2pi/log(5*100/(2pi)) ~ 2pi/log(80) ~ 6.28/4.4 ~ 1.43
# At T=2500: gap ~ 2pi/log(5*2500/(2pi)) ~ 2pi/log(2000) ~ 6.28/7.6 ~ 0.83
# Step ~ 0.02-0.05 should be safe and oversampled. Use adaptive: step=0.05 generally.
# 
# Let's first time a single Z evaluation.
import time
mp.mp.dps = 20
t0=time.time()
for _ in range(20):
 Z_chi5(mp.mpf('100.5'))
print('Z @ dps=20 t=100:', (time.time()-t0)/20, 'sec')

mp.mp.dps = 20
t0=time.time()
for _ in range(10):
 Z_chi5(mp.mpf('2400.5'))
print('Z @ dps=20 t=2400:', (time.time()-t0)/10, 'sec')


Z @ dps=20 t=100: 0.017150378227233885 sec


Z @ dps=20 t=2400: 0.21501753330230713 sec


In [11]:

# At dps=20, t=2400, ~0.2 s per eval. Gap ~0.84 means step 0.05 -> ~17 evals/zero per scan.
# Total scan evals = 2500/0.05 = 50,000. Single-threaded estimate:
# Time-weighted average ~0.1s per eval -> 5000s. Too slow single-thread.
# With multiprocessing on multiple cores it's feasible.
# Refining 5000 zeros at dps=50 takes additional time.
# 
# Let's first try dps=15 for the scan.
mp.mp.dps = 15
t0=time.time()
for _ in range(20):
 Z_chi5(mp.mpf('2400.5'))
print('Z @ dps=15 t=2400:', (time.time()-t0)/20, 'sec')

import os
print('CPUs:', os.cpu_count())


Z @ dps=15 t=2400: 0.16644341945648194 sec
CPUs: 8


In [12]:

# 8 CPUs available. mpmath's evaluation is the bottleneck. Let's see if we can speed up Lchi5
# by using Euler-Maclaurin on the L-series more directly. mp.dirichlet uses functional eq.
# Let me check if mpmath supports a faster L(s, chi) - it's already using best method internally.
#
# Alternative: use the approximate functional equation manually with a finite sum.
# For real even chi mod q, by AFE:
# L(1/2 + it, chi) = sum_{n<X} chi(n) n^{-1/2-it} 
# + eps * (q/pi)^(1/2 - it) * Gamma((1/2 - it)/2) / Gamma((1/2 + it)/2) 
# * sum_{n<Y} chi(n) n^{-1/2+it}
# where eps=1 and X*Y ~ q*t/(2*pi).
# A good practical formula. Let's just check if mp.dirichlet is using AFE; let's try our own.

# Use standard Riemann-Siegel-like approximation for L(s, chi):
# Z(t) = 2 * Re[ exp(i*theta(t)) * sum_{n=1}^{N} chi(n)/sqrt(n) * exp(-i*t*log n) ] + R
# where N = floor(sqrt(q*t/(2*pi))).
#
# For now, let's just use mp.dirichlet; 8-core parallelization should suffice.

# Let me write the parallel scan.
import numpy as np

def write_module():
 code = '''
import mpmath as mp

CHI = [0, 1, -1, -1, 1]
LOG_PI_OVER_5 = None
def theta_chi5(t):
 return mp.arg(mp.gamma(mp.mpf('0.25') + 1j*t/2)) - (t/2)*mp.log(mp.pi/5)

def Z_chi5_dps(t, dps=15):
 old = mp.mp.dps
 mp.mp.dps = dps
 try:
 t = mp.mpf(t)
 s = mp.mpc(mp.mpf('0.5'), t)
 L = mp.dirichlet(s, CHI)
 val = (mp.exp(1j*theta_chi5(t)) * L).real
 return float(val)
 finally:
 mp.mp.dps = old

def Z_chi5_high(t, dps=50):
 old = mp.mp.dps
 mp.mp.dps = dps
 try:
 t = mp.mpf(t)
 s = mp.mpc(mp.mpf('0.5'), t)
 L = mp.dirichlet(s, CHI)
 val = (mp.exp(1j*theta_chi5(t)) * L).real
 return val
 finally:
 mp.mp.dps = old

def scan_interval(args):
 """Scan [t_start, t_end] with step h at given dps. Return list of bracketing intervals (a,b,sa,sb)."""
 t_start, t_end, h, dps = args
 old = mp.mp.dps
 mp.mp.dps = dps
 try:
 t = mp.mpf(t_start)
 s = mp.mpc(mp.mpf('0.5'), t)
 L = mp.dirichlet(s, CHI)
 prev_z = float((mp.exp(1j*theta_chi5(t)) * L).real)
 prev_t = float(t_start)
 results = []
 cur = float(t_start) + h
 while cur <= t_end + 1e-12:
 t = mp.mpf(cur)
 s = mp.mpc(mp.mpf('0.5'), t)
 L = mp.dirichlet(s, CHI)
 z = float((mp.exp(1j*theta_chi5(t)) * L).real)
 if prev_z == 0.0 or z == 0.0 or (prev_z > 0) != (z > 0):
 results.append((prev_t, cur, prev_z, z))
 prev_z = z
 prev_t = cur
 cur += h
 return results
 finally:
 mp.mp.dps = old
'''
 with open('lchi5_module.py','w') as f:
 f.write(code)
write_module()
print('module written')


module written


In [13]:

# Quick test of one chunk of scan to estimate timing
import importlib, sys
if 'lchi5_module' in sys.modules: del sys.modules['lchi5_module']
import lchi5_module as M

t0 = time.time()
res = M.scan_interval((0.0, 50.0, 0.05, 15))
print(f'scan 0..50 took {time.time()-t0:.2f}s, found {len(res)} brackets')
print('first 5 brackets:', res[:5])


scan 0..50 took 6.15s, found 21 brackets
first 5 brackets: [(6.5999999999999845, 6.649999999999984, 0.07930849932084441, -0.002527871923894818), (9.800000000000004, 9.850000000000005, -0.050480700247246046, 0.029681411522162072), (11.950000000000035, 12.000000000000036, 0.019741555578106106, -0.09362286120213054), (16.000000000000092, 16.050000000000093, -0.07124376507655773, 0.033144096589426394), (17.550000000000114, 17.600000000000115, 0.027086264480608146, -0.053083175806870525)]


In [14]:

# Good - first bracket matches LMFDB 6.6484... Found 21 brackets in [0,50] - expected ~17 zeros there.
# Wait, 21 brackets > 21 zeros. Possibly each bracket = 1 zero. Let me count from LMFDB:
# Positive zeros up to 50:
zeros_str = data['positive_zeros']
zeros_under_50 = [float(z) for z in zeros_str if float(z) < 50]
print(len(zeros_under_50), 'LMFDB zeros under 50')
print(zeros_under_50)


21 LMFDB zeros under 50
[6.648453344727715, 9.83144443288667, 11.958845626083514, 16.033821128384236, 17.566994292325557, 19.540732622784752, 22.22740545445941, 24.588466217408197, 26.77609594800414, 28.461035100177522, 29.707909350480964, 33.000456006870515, 34.728812978904806, 35.868638371812274, 38.12918472143653, 39.560572946403184, 41.842438545791694, 44.031290061441695, 45.42730008278229, 46.49272715949141, 48.34566182106784]


In [15]:

# Perfect - 21 zeros under 50 matches our 21 brackets. Excellent.
# Now estimate total time for full scan up to T=2500.
# At t=2400, ~0.2s per eval at dps=15-20. Scan h=0.05 means 50,000 evals total but more density at high t.
# Let me estimate: integral of cost.
# 
# We'll use h=0.05 for safety. Total evals ~ 2500/0.05 = 50000. 
# At t=0: 0.005s, at t=2500: ~0.2s. Linear-ish weighted average: ~0.1s.
# Total single-thread ~5000s. With 8 cores: ~625s ~ 10 min. Acceptable.
#
# But h=0.05 may be too coarse near t=2500 where gap can be as small as 0.84 but variability exists.
# Use adaptive: use h proportional to expected gap. Smaller h at high t.
# Actually 0.05 << 0.83 so fine. Use 0.05 everywhere.

# Parallelize: split [0, 2500] into chunks of ~25 units each (100 chunks).
# Each chunk: scan_interval. Pool of 8.

# Let me first do a smaller test then scale.
from multiprocessing import Pool

if __name__ == '__main__':
 pass

# Make sure module is importable from worker
chunks_test = [(i*10.0, (i+1)*10.0, 0.05, 15) for i in range(5)]
t0 = time.time()
with Pool(4) as pool:
 out = pool.map(M.scan_interval, chunks_test)
print(f'parallel test 50 units took {time.time()-t0:.1f}s')
total = sum(len(o) for o in out)
print('total brackets:', total)


parallel test 50 units took 3.1s
total brackets: 21


In [16]:

# Good - 4 workers on 5 chunks of 10 units took 3.1s. Sequential would be ~6s. 
# Note: at low t, mp.dirichlet is fast. At high t, mp.dirichlet uses functional equation involving sums.
# Let's measure cost vs t to better estimate.
import lchi5_module as M2
mp.mp.dps = 15
for tval in [10, 100, 500, 1000, 1500, 2000, 2500]:
 t0 = time.time()
 M2.scan_interval((tval, tval+5.0, 0.05, 15))
 print(f't={tval}: {time.time()-t0:.2f}s per 5 units (100 evals)')


t=10: 0.51s per 5 units (100 evals)


t=100: 1.46s per 5 units (100 evals)


t=500: 5.58s per 5 units (100 evals)


t=1000: 9.56s per 5 units (100 evals)


t=1500: 9.53s per 5 units (100 evals)


t=2000: 16.85s per 5 units (100 evals)


t=2500: 16.59s per 5 units (100 evals)


In [17]:

# Costs per 5 units: 0.51 + 1.46 + 5.58 + 9.56 + 9.53 + 16.85 + 16.59 = 60s for sampled 35 units
# Actual full scan integral: roughly piecewise:
# 0-50: ~6s ; 50-200: ~50s ; 200-500: ~250s; 500-1000: ~800s; 1000-1500: ~900s; 1500-2000: ~1500s; 2000-2500: ~1650s
# Total single-thread ~ 5000s. With 8-way parallel: ~625s = ~10.4 min. Doable.
# Plus refinement of 5000 zeros at dps=50: ~5000 * ~0.5s each = 2500s/8 = 5 min.
# Total ~15-20 min. Within budget.

# Actually let me re-evaluate. mp.dirichlet at dps=15 takes very long at high t.
# Maybe I should reduce step a bit but increase efficiency. 
# Let me try writing a faster approximate functional equation directly.

# Actually let me check: maybe a smaller step lets us skip refinement effort.
# Or maybe a larger step is OK since gap >= 0.84. h=0.1 gives 25,000 evals -> half the time.
# Risk: missing closely-spaced zeros. At chi mod 5 these are well-separated. Use h=0.1.

mp.mp.dps = 15
import lchi5_module as M3
t0=time.time()
out = M3.scan_interval((100, 200, 0.1, 15))
print(f'scan 100-200 h=0.1: {time.time()-t0:.1f}s, found {len(out)} brackets')

# Sanity check with h=0.05:
t0=time.time()
out2 = M3.scan_interval((100, 200, 0.05, 15))
print(f'scan 100-200 h=0.05: {time.time()-t0:.1f}s, found {len(out2)} brackets')

# Should match
print('Match count:', len(out)==len(out2))


scan 100-200 h=0.1: 20.9s, found 75 brackets


scan 100-200 h=0.05: 41.7s, found 75 brackets
Match count: True


In [18]:

# Both find 75 brackets with h=0.1 and h=0.05 in [100,200]. Good.
# But careful: at higher t, gap shrinks. Min gap at T=2500 is ~0.84, but a few zeros may be close.
# h=0.1 is risky but probably safe for chi mod 5 (well-separated). 
# Let me test at high t.

t0=time.time()
out_a = M3.scan_interval((2000, 2050, 0.1, 15))
print(f'h=0.1: {time.time()-t0:.1f}s, {len(out_a)} brackets')
t0=time.time()
out_b = M3.scan_interval((2000, 2050, 0.05, 15))
print(f'h=0.05: {time.time()-t0:.1f}s, {len(out_b)} brackets')

# Compare
print('h=0.1:', [b[0] for b in out_a[:5]])
print('h=0.05:', [b[0] for b in out_b[:5]])
print('counts match:', len(out_a)==len(out_b))


h=0.1: 83.0s, 59 brackets


h=0.05: 165.8s, 59 brackets
h=0.1: [2000.7999999999993, 2000.8999999999992, 2002.099999999998, 2002.4999999999977, 2003.299999999997]
h=0.05: [2000.7999999999993, 2000.9499999999991, 2002.099999999998, 2002.5499999999977, 2003.349999999997]
counts match: True


In [19]:

# h=0.1 finds same brackets. Use h=0.1.
# Let me consider an even larger step. h=0.2 still safe since min gap > 0.8.
# But h=0.2 may miss 2 close zeros if gap < 0.4. Probably safe but risky.
# Stick with h=0.1.
#
# Now, I realize at high t, the cost per Z eval is dominated by mp.dirichlet which internally
# evaluates ~sqrt(qt/(2pi)) terms. That's ~sqrt(2500*5/6.28) ~ ~45 terms at t=2500. 
# mp.dirichlet should be quite fast. Let me verify by trying my own AFE-based code.

import mpmath as mp
mp.mp.dps = 15

def Z_chi5_afe(t, dps=15):
 """Approximate functional equation for L(s, chi_5)."""
 mp.mp.dps = dps
 t = mp.mpf(t)
 q = 5
 s = mp.mpc(mp.mpf('0.5'), t)
 # AFE: L(s,chi) = sum_{n<=X} chi(n)*n^{-s} + epsilon * G(s) * sum_{n<=Y} chi(n)*n^{-(1-s)}
 # where G(s) = (q/pi)^(1/2 - s) * Gamma((1-s)/2)/Gamma(s/2) (for a=0, even)
 # and X*Y = q/(2pi) * |t|... actually X = Y = sqrt(qt/(2pi)) typical truncation
 X = int(mp.sqrt(q*t/(2*mp.pi))) + 10
 
 chi = [0, 1, -1, -1, 1]
 sum1 = mp.mpc(0)
 sum2 = mp.mpc(0)
 for n in range(1, X+1):
 c = chi[n % 5]
 if c == 0: continue
 ln = mp.log(n)
 sum1 += c * mp.exp(-s * ln)
 sum2 += c * mp.exp(-(1-s) * ln)
 # G(s)
 G = mp.power(mp.mpf(q)/mp.pi, mp.mpf('0.5') - s) * mp.gamma((1-s)/2) / mp.gamma(s/2)
 L = sum1 + G * sum2
 # Z(t) = exp(i theta(t)) L(1/2+it, chi)
 th = mp.arg(mp.gamma(mp.mpf('0.25') + 1j*t/2)) - (t/2)*mp.log(mp.pi/q)
 return float((mp.exp(1j*th) * L).real)

# Compare with mp.dirichlet
mp.mp.dps = 25
val_dir = float(M3.Z_chi5_dps(100.5, 20))
val_afe = Z_chi5_afe(100.5, 25)
print('dirichlet:', val_dir)
print('AFE:', val_afe)
print('diff:', val_afe - val_dir)


dirichlet: 1.5663165213788304
AFE: 0.823442395868003
diff: -0.7428741255108273


In [20]:

# AFE doesn't match. There's a balanced AFE issue. The truncation needs corrections.
# Let me skip my AFE attempt and just rely on mp.dirichlet — it works correctly and is reasonably fast.
# Let's also try a faster Dirichlet L direct evaluation method.
# Actually let's just commit to using mp.dirichlet with h=0.1 step and 8 workers.

# Also let me try mpmath's lerchphi or zeta(s, chi) -- mp.dirichlet should suffice.

# Quick sanity at higher dps:
mp.mp.dps = 50
zero1 = mp.mpf('6.64845334472771471612327845997')
s = mp.mpc('0.5', zero1)
L = mp.dirichlet(s, [0,1,-1,-1,1])
print('|L| at zero1 (dps=50):', abs(L))
print('mp.dirichlet at high t and dps=50:')
t0=time.time()
for _ in range(3):
 mp.dirichlet(mp.mpc('0.5', '2400.5'), [0,1,-1,-1,1])
print('time per call dps=50, t=2400:', (time.time()-t0)/3)


|L| at zero1 (dps=50): 1.5230029044760436910112625234985957925733108250633e-29
mp.dirichlet at high t and dps=50:


time per call dps=50, t=2400: 0.3152011235555013


In [21]:

# At dps=50, t=2400, one eval is 0.32s. Refining 5000 zeros (each ~5-10 evals via findroot) = 5000*7*0.3 = 10,500s.
# That's too slow. Need parallelize refinement too. Plus also use brent on float bracket.
# 
# Better approach: refine using mp.findroot with solver='anderson' or 'bisect' starting from
# a tight bracket. With a tight bracket, just a few mp.dirichlet calls. 
# At low t, refinement is cheap. At high t (~2400), ~5 calls * 0.3s = 1.5s/zero.
# Average maybe 0.4s/zero -> 2000s/8 = 250s. OK.

# Let's now build the actual end-to-end pipeline.

# Refactor module
code = '''
import mpmath as mp

CHI = [0, 1, -1, -1, 1]

def theta_chi5(t):
 return mp.arg(mp.gamma(mp.mpf('0.25') + 1j*t/2)) - (t/2)*mp.log(mp.pi/5)

def Z(t):
 t = mp.mpf(t)
 s = mp.mpc(mp.mpf('0.5'), t)
 L = mp.dirichlet(s, CHI)
 return (mp.exp(1j*theta_chi5(t)) * L).real

def scan_interval(args):
 """Scan [t_start, t_end] with step h at given dps. Return list of (a,b) brackets with sign change."""
 t_start, t_end, h, dps = args
 mp.mp.dps = dps
 results = []
 cur = float(t_start)
 prev_z = float(Z(cur))
 prev_t = cur
 cur += h
 while cur <= t_end + 1e-12:
 z = float(Z(cur))
 if z == 0.0 or (prev_z > 0) != (z > 0):
 results.append((prev_t, cur))
 prev_z = z
 prev_t = cur
 cur += h
 return results

def refine_bracket(args):
 """Refine a bracket (a,b) to a zero at target dps using mp.findroot."""
 a, b, target_dps = args
 mp.mp.dps = target_dps + 5
 a_mp = mp.mpf(a)
 b_mp = mp.mpf(b)
 # Use bisection-style mp.findroot with 'bisect' solver
 try:
 # Use 'anderson' (secant) with initial bracket midpoint, refine carefully
 root = mp.findroot(Z, (a_mp, b_mp), solver='anderson')
 except Exception:
 # Fallback: bisection
 za = Z(a_mp)
 zb = Z(b_mp)
 for _ in range(200):
 m = (a_mp + b_mp) / 2
 zm = Z(m)
 if zm == 0 or (b_mp - a_mp) < mp.mpf(10)**(-target_dps-3):
 root = m
 break
 if (zm > 0) == (za > 0):
 a_mp = m
 za = zm
 else:
 b_mp = m
 zb = zm
 else:
 root = (a_mp + b_mp) / 2
 return mp.nstr(root, target_dps+2, strip_zeros=False)

def refine_chunk(args):
 """Refine a chunk of brackets. args = (brackets_list, target_dps)."""
 brackets, target_dps = args
 return [refine_bracket((a, b, target_dps)) for (a, b) in brackets]
'''
with open('lchi5_module.py','w') as f:
 f.write(code)

# reload
if 'lchi5_module' in sys.modules: del sys.modules['lchi5_module']
import lchi5_module as M
print('module rewritten')


module rewritten


In [22]:

# Test the refinement on first zero
t0 = time.time()
r = M.refine_bracket((6.6, 6.7, 50))
print('refined first zero:', r)
print('LMFDB: 6.64845334472771471612327845997')
print('time:', time.time()-t0)


refined first zero: 6.648453344727714716123278459979317847298585423244498
LMFDB: 6.64845334472771471612327845997
time: 0.1798698902130127


In [23]:

# Excellent! Matches LMFDB to 29 digits. dps=50 has 50 digits.
# Now run the full scan.
# Split [0, 2500] into 50 chunks of 50 units each, parallel with 8 workers.

from multiprocessing import Pool
chunks = []
T_max = 2500.0
chunk_len = 50.0 # per chunk
h = 0.1
dps_scan = 15
n_chunks = int(T_max/chunk_len)
for i in range(n_chunks):
 a = i*chunk_len
 b = (i+1)*chunk_len
 # First chunk: start at small epsilon to avoid t=0
 if i == 0:
 a = 0.5
 chunks.append((a, b, h, dps_scan))
print(f'{len(chunks)} chunks of length {chunk_len}, h={h}, dps_scan={dps_scan}')

# Time it (this is the big run)
t0 = time.time()
with Pool(8) as pool:
 all_results = pool.map(M.scan_interval, chunks)
elapsed = time.time() - t0
print(f'SCAN elapsed: {elapsed:.1f}s = {elapsed/60:.1f} min')

# Flatten brackets in order
brackets = []
for r in all_results:
 brackets.extend(r)
print(f'Total brackets found: {len(brackets)}')
print('first 5:', brackets[:5])
print('last 5:', brackets[-5:])


50 chunks of length 50.0, h=0.1, dps_scan=15


SCAN elapsed: 549.0s = 9.1 min
Total brackets found: 2622
first 5: [(6.5999999999999925, 6.699999999999992), (9.799999999999981, 9.89999999999998), (11.899999999999974, 11.999999999999973), (15.99999999999996, 16.09999999999996), (17.49999999999998, 17.59999999999998)]
last 5: [(2495.5999999999585, 2495.6999999999584), (2496.7999999999574, 2496.8999999999573), (2498.299999999956, 2498.399999999956), (2498.6999999999557, 2498.7999999999556), (2499.399999999955, 2499.499999999955)]


In [24]:

# Only found 2622 brackets - need 5000! Let me check. Estimate at T=2500 gives N~5249.
# So scan should have found ~5249. We got 2622, about half. 
# Issue: h=0.1 may be too coarse. Let me check at high t.
# Actually, more likely: I'm checking only zero crossings, and Z(t) can stay sign-constant over even 
# numbered crossings cancelling? No, sign change = 1 zero (generically).
# 
# Wait — for a Dirichlet L-function with real coefficients, Z(t) is real on the critical line.
# Multiple zeros (double zeros) would not show sign change but should be rare.
# Let me sanity check: how many zeros up to T=200?
print(len([z for z in data['positive_zeros'] if float(z) < 200]))
# 129 zeros - this is the entire LMFDB list, all up to ~199.

# vMangoldt at T=200:
print('Vmangoldt N(200):', Napprox(200))
# So ~129 zeros up to 200. That matches LMFDB.

# Our scan found:
# Up to T=200:
under_200 = [b for b in brackets if b[1] < 200]
print(f'Our brackets under T=200: {len(under_200)}')
# Up to T=2500:
print(f'Total under 2500: {len(brackets)}')


129
Vmangoldt N(200): 259.09649412518695
Our brackets under T=200: 129
Total under 2500: 2622


In [25]:

# Interesting! Under T=200: 129 brackets matches LMFDB exactly. 
# But Napprox(200)=259 suggests ~259 zeros! That's twice what LMFDB has.
# Wait — that approximation N(T) = (T/pi)*log(qT/(2*pi*e)) is for an even character mod q.
# Let me reread:
# For Dirichlet L: N(T, chi) ~ (T/2pi)*log(qT/(2*pi)) - (T/2pi) + O(log T) (counting zeros 0 <= Im(rho) <= T)
# Hmm let me re-derive.
# For chi mod q non-trivial (a=0), N(T) = (T/2π)*log(qT/(2π)) - T/(2π) + O(log T)
# = (T/(2π))*[log(qT/(2π)) - 1] = (T/(2π))*log(qT/(2πe))
# 
# So N(200) = (200/(2π))*log(5*200/(2π*e)) = (200/6.28)*log(1000/17.08) = 31.83 * log(58.6) = 31.83 * 4.07 = 129.5
# 
# I had the formula wrong (off by factor 2)! Let me redo:
def Napprox2(T, q=5):
 return (T/(2*math.pi))*math.log(q*T/(2*math.pi*math.e))
print(Napprox2(200)) # should be ~130
print(Napprox2(2500)) # to estimate
# So my approximation was off. Real T_max for N=5000 is much larger.
from scipy.optimize import brentq
T_for_5000 = brentq(lambda T: Napprox2(T)-5000, 100, 50000)
print('T for N=5000:', T_for_5000)


129.54824706259348
2624.3085849085446
T for N=5000: 4388.709742982143


In [26]:

# T for N=5000 is ~4400. Our scan up to 2500 found 2622 zeros (matches Napprox2(2500)=2624).
# Need to scan up to T ~ 4500 to be safe.
# Already scanned 0-2500. Continue from 2500 to 4500.

# Existing brackets count: 2622. Need 2378 more. They are between T=2500 and T~4400.
# Let me run additional 2500-4500.

chunks2 = []
T_start = 2500.0
T_end = 4500.0
for i in range(int((T_end - T_start)/chunk_len)):
 a = T_start + i*chunk_len
 b = a + chunk_len
 chunks2.append((a, b, h, dps_scan))
print(f'{len(chunks2)} additional chunks')

t0 = time.time()
with Pool(8) as pool:
 more_results = pool.map(M.scan_interval, chunks2)
elapsed = time.time() - t0
print(f'SCAN2 elapsed: {elapsed:.1f}s = {elapsed/60:.1f} min')
more_brackets = []
for r in more_results:
 more_brackets.extend(r)
print(f'Additional brackets: {len(more_brackets)}')

all_brackets = brackets + more_brackets
print(f'Total brackets: {len(all_brackets)}')


TimeoutError: Code execution timed out after 900.0 seconds

In [27]:

# Timed out. Need a different approach - smaller chunks, save progress.
# Let me check what we have so far. Probably the pool was running.
# Save 'brackets' (2622 from 0-2500) to disk first.
import pickle
with open('brackets_0_2500.pkl', 'wb') as f:
 pickle.dump(brackets, f)
print('Saved', len(brackets), 'brackets to disk')


Saved 2622 brackets to disk


In [28]:

# At t=2500-4500, evaluations are more expensive. Need to chunk differently and persist.
# Time per chunk at t~3000-4500 increased a lot. Let's measure.
mp.mp.dps = 15
t0=time.time()
out_test = M.scan_interval((4000, 4005, 0.1, 15))
print(f'scan 4000-4005 (50 evals) h=0.1: {time.time()-t0:.1f}s, {len(out_test)} brackets')


scan 4000-4005 (50 evals) h=0.1: 15.8s, 6 brackets


In [29]:

# 0.32s per eval at t=4000. For 0.1 step: scan 2500-4500 = 20000 evals * 0.32 = 6400s single thread = 800s / 8 cores = ~13min.
# That should fit. The previous attempt timed out at 900s, probably it was close.
# Issue was the cell-level timeout. Let me split into smaller batches with progress dumps.

# Approach: scan in 5 chunks of 400 units, save after each.
import os
chunks_pending = []
T_start = 2500.0
T_end = 4500.0
batch_size = 200.0 # process 200 units per batch (400-500s estimate)
batch_starts = [T_start + i*batch_size for i in range(int((T_end-T_start)/batch_size))]
print('Batches start at:', batch_starts)


Batches start at: [2500.0, 2700.0, 2900.0, 3100.0, 3300.0, 3500.0, 3700.0, 3900.0, 4100.0, 4300.0]


In [30]:

# Run batch 1: 2500-2700
def run_batch(t_lo, t_hi, h=0.1, dps_scan=15, chunk_len=25.0, workers=8):
 chunks_local = []
 n = int(round((t_hi - t_lo)/chunk_len))
 for i in range(n):
 a = t_lo + i*chunk_len
 b = a + chunk_len
 chunks_local.append((a, b, h, dps_scan))
 t0 = time.time()
 with Pool(workers) as pool:
 results = pool.map(M.scan_interval, chunks_local)
 elapsed = time.time()-t0
 out = []
 for r in results:
 out.extend(r)
 print(f'batch {t_lo}-{t_hi}: {elapsed:.1f}s, {len(out)} brackets')
 return out

b_2500_2700 = run_batch(2500, 2700)
with open('brackets_2500_2700.pkl', 'wb') as f:
 pickle.dump(b_2500_2700, f)


batch 2500-2700: 72.4s, 243 brackets


In [31]:

b_2700_2900 = run_batch(2700, 2900)
with open('brackets_2700_2900.pkl','wb') as f: pickle.dump(b_2700_2900, f)
b_2900_3100 = run_batch(2900, 3100)
with open('brackets_2900_3100.pkl','wb') as f: pickle.dump(b_2900_3100, f)


batch 2700-2900: 72.9s, 246 brackets


batch 2900-3100: 72.9s, 247 brackets


In [32]:

b_3100_3300 = run_batch(3100, 3300)
with open('brackets_3100_3300.pkl','wb') as f: pickle.dump(b_3100_3300, f)
b_3300_3500 = run_batch(3300, 3500)
with open('brackets_3300_3500.pkl','wb') as f: pickle.dump(b_3300_3500, f)


batch 3100-3300: 72.5s, 250 brackets


batch 3300-3500: 72.9s, 252 brackets


In [33]:

# Need to keep moving. Continue scanning.
b_3500_3700 = run_batch(3500, 3700)
with open('brackets_3500_3700.pkl','wb') as f: pickle.dump(b_3500_3700, f)
b_3700_3900 = run_batch(3700, 3900)
with open('brackets_3700_3900.pkl','wb') as f: pickle.dump(b_3700_3900, f)


batch 3500-3700: 123.7s, 253 brackets


batch 3700-3900: 132.5s, 255 brackets


In [34]:

# Check progress. Total so far:
total_so_far = 2622 + 243 + 246 + 247 + 250 + 252 + 253 + 255
print('total brackets so far:', total_so_far)
print('Need 5000. Remaining:', 5000 - total_so_far)
# Need ~632 more. Continue scanning 3900-4400.


total brackets so far: 4368
Need 5000. Remaining: 632


In [35]:

# Need to scan 3900-4500 (~750 zeros) and refine 5000 zeros to dps=50.
# Time constraints critical. Let's parallelize remaining scan + start refinement in parallel where possible.
# Run 3900-4500 in one batch.
b_3900_4500 = run_batch(3900, 4500, chunk_len=25.0)
with open('brackets_3900_4500.pkl','wb') as f: pickle.dump(b_3900_4500, f)
print('Done 3900-4500')


TimeoutError: Code execution timed out after 219.0 seconds